In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 3"
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'
from tqdm import tqdm
import random
random.seed(1337)
import matplotlib.pyplot as plt
import argparse
import numpy as np
np.random.seed(1337)
import pandas as pd
import torch
import sys

from src.mrl_te_optimization.framepool import *
from src.mrl_te_optimization.util import *
from utilities.UTR3_utilities import Seq2Tensor, Condition2Tensor, CELLTYPE_CODES_UTR3, CELLTYPE_UTR3, predict, make_res, UTRData
import keras
from utilities.pl_regressor import RNARegressor
from utilities.UTR3_utilities_optimization import MinGapEnergy
import random
random.seed(1337)

from utilities.legnet_classifier import LegNetClassifier
import utilities.legnet_classifier as legnet_classifier
sys.modules['legnet_classifier'] = legnet_classifier

import scipy.stats as stats


from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from torch.utils.data import DataLoader, Dataset
tf.compat.v1.enable_eager_execution()

gpu_options = tf.compat.v1.GPUOptions(per_process_gpu_memory_fraction=0.75)

sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(gpu_options=gpu_options))

DATA = 'data/UTR3_zscores_replicateagg.csv'
BATCH_SIZE = 64
STEPS = 10


print(tf.test.gpu_device_name())

DIM = 80
SEQ_LEN = 240
UTR_LEN = 240
gpath = './models/checkpoint_3000.h5'
model_path = f"utilities/model-utr3-deltas-epoch=9-step=1330.ckpt"


out_folder = './outputs/'
os.makedirs(out_folder, exist_ok=True)

2026-05-01 19:12:34.801885: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 19:12:34.869379: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/mnt/calc/skaraban_smtb/.conda/envs/utrgan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-01 19:12:36.188334: I tensorfl

/device:GPU:0


2026-05-01 19:12:40.061027: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 34093 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:07:00.0, compute capability: 8.9
2026-05-01 19:12:40.062367: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 34093 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:c3:00.0, compute capability: 8.9
2026-05-01 19:12:40.091797: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /device:GPU:0 with 34093 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:07:00.0, compute capability: 8.9
2026-05-01 19:12:40.092681: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /device:GPU:1 with 34093 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:c3:00.0, compute capability: 8.9


In [2]:
# LOADING WGAN FROM CHECKPOINTS 
from src.gan.lib import models
N_CHANNELS = DIM

gen_layers = 3
disc_layers = 3

LR = 1e-3


G = models.resnet_g2(DIM, N_CHANNELS, SEQ_LEN, 5, res_layers=gen_layers)
D = models.resnet_d2(N_CHANNELS, SEQ_LEN, 5, res_layers=disc_layers)

G_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)
D_optimizer = Adam(learning_rate=LR, beta_1=0.5, beta_2=0.99)


checkpoint = tf.train.Checkpoint(generator=G,
                                 discriminator=D,
                                 g_optimizer=G_optimizer,
                                 d_optimizer=D_optimizer)


checkpoint_dir = "checkpoints/tensorflow/WGAN-TF2-UTR3-oversampled"

latest_ckpt = tf.train.latest_checkpoint(checkpoint_dir)

status = checkpoint.restore(latest_ckpt)

status.expect_partial() 
print(f"Model restored from {latest_ckpt}")

G.summary()
D.summary()


2026-05-01 19:12:40.127263: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 34093 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:07:00.0, compute capability: 8.9
2026-05-01 19:12:40.128150: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 34093 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:c3:00.0, compute capability: 8.9


Model restored from checkpoints/tensorflow/WGAN-TF2-UTR3-oversampled/ckpt-8
Model: "Generator"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 80)]         0           []                               
                                                                                                  
 dense (Dense)                  (None, 19200)        1555200     ['input_1[0][0]']                
                                                                                                  
 reshape (Reshape)              (None, 240, 80)      0           ['dense[0][0]']                  
                                                                                                  
 activation (Activation)        (None, 240, 80)      0           ['reshape[0][0]']                
              

In [3]:
# LOADING PREDICTOR 
device = 'cuda:0'
model_predict = RNARegressor.load_from_checkpoint(model_path, map_location='cpu').model.to(device)
model_predict.eval()
input_len = 240

In [4]:
total_params = sum(p.numel() for p in model_predict.parameters())
print("Total parameters:", total_params)

trainable_params = sum(p.numel() for p in model_predict.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)


Total parameters: 269066
Trainable parameters: 269066


In [5]:
# PARAMETERS
BATCH_SIZE = 128
DIM = 240
STEPS = 10
LR = 1e-3
CELL_LINE = "c1" 
SEQ_LEN = 240

In [ ]:
rna_vocab = {"A":0,
             "C":1,
             "G":2,
             "T":3,
             "*":4}

rev_rna_vocab = {v:k for k,v in rna_vocab.items()}

In [6]:
def recover_seq(samples, rev_charmap=rev_rna_vocab):
    """Convert samples to strings and save to log directory."""
    
    char_probs = samples
    argmax = np.argmax(char_probs, 2)
    seqs = []
    for line in argmax:
        s = "".join(rev_charmap[d] for d in line)
        seqs.append(s)
    return seqs 

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_addons as tfa
from tqdm import tqdm
import gc
import logging 
import torch

dataset_paths = {
    "c1": "data/scored_UTR3_80dim_10k_c1.csv",
    "c2": "data/scored_UTR3_80dim_10k_c2.csv",
    "c4": "data/scored_UTR3_80dim_10k_c4.csv",
    "c6": "data/scored_UTR3_80dim_10k_c6.csv",
    "c13": "data/scored_UTR3_80dim_10k_c13.csv",
    "c17": "data/scored_UTR3_80dim_10k_c17.csv",
}


def noise_to_tensor(noise_str: str):
    clean = noise_str[1:-1].replace("\n", " ").strip()
    values = [float(x) for x in clean.split() if x]
    return torch.tensor(values, dtype=torch.float32)

logging.basicConfig(
    filename="optlog.log",
    filemode="a",  
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
)

def optimize_batch(latent_batch, cell_line, start, end):
    logging.info(f"=== Starting batch {start}-{end} for {cell_line} ===")

    try:
        initial_lr = 1e-2
        max_lr = 1e-4
        step_size = 1000

        cycleLR = tfa.optimizers.CyclicalLearningRate(
            initial_learning_rate=initial_lr,
            maximal_learning_rate=max_lr,
            step_size=step_size,
            scale_fn=lambda x: 1.0,
            scale_mode="cycle",
        )

        noise = tf.Variable(latent_batch, dtype=tf.float32)
        optimizer = tf.keras.optimizers.Adam(cycleLR)

        seq2tensor = Seq2Tensor()
        c2t = Condition2Tensor(
            num_conditions=len(CELLTYPE_CODES_UTR3),
            celltype_codes=CELLTYPE_CODES_UTR3,
        )

        energy_fn = MinGapEnergy(
            model=model_predict,
            device=device,
            seq_len=SEQ_LEN,
            target_feature=CELLTYPE_UTR3[cell_line],
            target_alpha=1.0,
            bending_factor=0.0,
            loss_type="malinois",
        )

        best_energy, best_seqs = None, None

        import time 

        for step in tqdm(range(STEPS)):
            if step % 100 == 0:
                logging.debug(f"Step {step}...")

            with tf.GradientTape() as gen_tape:
                gen_out = G(noise)
                seqs_gen = recover_seq(gen_out.numpy(), rev_rna_vocab)
                seqs_tf = gen_out[:, :, :4]  
                seqs_torch = torch.from_numpy(seqs_tf.numpy()).permute(0,2,1).float().to(device)
                seqs_torch.requires_grad_(True)
                energy = energy_fn(seqs_torch)
                mean_energy = energy.mean()
                mean_energy.backward()

                pt_grads = seqs_torch.grad.detach().cpu().numpy()
                pt_grads = np.pad(pt_grads, ((0,0),(0,1),(0,0)))
                tf_grads = tf.convert_to_tensor(pt_grads, dtype=tf.float32)
                tf_grads = tf.transpose(tf_grads, [0, 2, 1])

                gen_grads = gen_tape.gradient(gen_out, noise, output_gradients=tf_grads)
                optimizer.apply_gradients([(gen_grads, noise)])

            current_energy = mean_energy.item()
            if step % 100 == 0:
                logging.info(f"Batch {start}-{end}, step {step}: mean_energy={current_energy:.4f}")

            if best_energy is None or current_energy < best_energy:
                best_energy = current_energy
                best_seqs = seqs_gen

        logging.info(f"=== Finished batch {start}-{end} for {cell_line}, best_energy={best_energy:.4f} ===")

    except Exception as e:
        logging.exception(f"Error in batch {start}-{end} for {cell_line}: {e}")
        raise

    finally:
        del noise, optimizer, energy, mean_energy, gen_out, seqs_torch
        torch.cuda.empty_cache()
        gc.collect()

    return best_seqs, best_energy

/mnt/calc/skaraban_smtb/.conda/envs/utrgan/lib/python3.9/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/mnt/calc/skaraban_smtb/.conda/envs/utrgan/lib/python3.9/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.11.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter 

In [ ]:
STEPS = 4000
TOP_K = 2000
BATCH_SIZE = 32

cell_lines = ["c1", "c2", "c4", "c6", "c13", "c17"]

for cell_line in cell_lines:
    print(f"Loading dataset for {cell_line}...")
    dataset = pd.read_csv(dataset_paths[cell_line])
    dataset["latent_vector"] = dataset["latent_vector"].apply(noise_to_tensor)

    top_indices = dataset["score"].nlargest(TOP_K).index
    top_latent_vectors_all = np.stack(dataset.loc[top_indices, "latent_vector"].values)

    out_file = f"data/optimized_seqs_UTR3_{cell_line}.txt"
    with open(out_file, "w") as f_out:
        for start in range(0, TOP_K, BATCH_SIZE):
            end = min(start + BATCH_SIZE, TOP_K)
            latent_batch = top_latent_vectors_all[start:end]

            best_seqs, best_energy = optimize_batch(latent_batch, cell_line, start, end)

            logging.info(f"Best sequences for {cell_line} batch {start}-{end}, best energy: {best_energy}")
            for seq in best_seqs:
                f_out.write(f"{seq}\n")
